![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 4 -- Lab 2: Sentiment Analysis -- from scratch vs. pretrained

Is a movie review **positive** or **negative**? We'll solve it two ways and
compare:

1. Build everything **from scratch** -- a BPE tokenizer and an LSTM.
2. Reuse a **pretrained BERT** and just train a small classifier on top.

Which one wins with only a few thousand examples?

## 1) Setup

In [ ]:
%pip install datasets transformers -q

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset

## 2) Get the data

Short movie-review snippets labelled positive (1) or negative (0).

In [ ]:
data = load_dataset("rotten_tomatoes")
train_texts, train_labels = data["train"]["text"], data["train"]["label"]
test_texts,  test_labels  = data["test"]["text"],  data["test"]["label"]
print(len(train_texts), "train /", len(test_texts), "test")

# Part 1 -- From scratch: BPE tokenizer + LSTM

## 3) Train a BPE tokenizer

BPE learns common chunks of text (subwords) from the reviews and gives each a
number.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tok = Tokenizer(models.BPE(unk_token="[UNK]"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=5000, special_tokens=["[PAD]", "[UNK]"])
tok.train_from_iterator(train_texts, trainer)
print("vocab size:", tok.get_vocab_size())

## 4) Turn text into fixed-length number sequences

In [ ]:
PAD, MAXLEN = tok.token_to_id("[PAD]"), 40

def encode(texts):
    seqs = []
    for t in texts:
        ids = tok.encode(t).ids[:MAXLEN]
        ids += [PAD] * (MAXLEN - len(ids))       # pad short reviews
        seqs.append(ids)
    return torch.tensor(seqs)

Xtr, ytr = encode(train_texts), torch.tensor(train_labels)
Xte, yte = encode(test_texts),  torch.tensor(test_labels)

## 5) Build the LSTM model

Embedding (word -> vector) -> LSTM (reads the sequence) -> Linear (positive/negative).

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab, dim=64, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=PAD)
        self.lstm = nn.LSTM(dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 2)
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))       # h[-1] = final summary of the review
        return self.fc(h[-1])

model = SentimentLSTM(tok.get_vocab_size())

## 6) Train it

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(5):
    for xb, yb in dl:
        opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
    print(f"epoch {epoch + 1} done")

## 7) How accurate is it?

In [ ]:
model.eval()
with torch.no_grad():
    lstm_acc = (model(Xte).argmax(1) == yte).float().mean().item()
print(f"LSTM accuracy: {lstm_acc:.1%}")

## 8) Try your own sentence

In [ ]:
def predict(text):
    with torch.no_grad():
        pred = model(encode([text])).argmax(1).item()
    return "positive" if pred == 1 else "negative"

print(predict("An absolute masterpiece, I loved every minute."))
print(predict("Boring, predictable and a total waste of time."))

# Part 2 -- Pretrained: frozen BERT + a classifier head

Instead of learning language from zero, we reuse **BERT**, which already
understands English from huge amounts of text. We **freeze** it, turn each review
into an embedding, and train a tiny classifier on top -- classic transfer
learning.

## 9) Load a frozen BERT

In [ ]:
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
bert_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert-base-uncased").to(device).eval()

## 10) Turn each review into a BERT embedding

We average BERT's output over the words to get one vector per review.

In [ ]:
@torch.no_grad()
def embed(texts):
    out = []
    for i in range(0, len(texts), 64):
        batch = bert_tok(texts[i:i + 64], padding=True, truncation=True,
                         max_length=40, return_tensors="pt").to(device)
        h = bert(**batch).last_hidden_state          # one vector per word
        mask = batch["attention_mask"].unsqueeze(-1)
        out.append(((h * mask).sum(1) / mask.sum(1)).cpu())   # average the words
    return torch.cat(out).numpy()

train_emb, test_emb = embed(train_texts), embed(test_texts)
print("embeddings:", train_emb.shape)

## 11) Train a simple classifier head on the embeddings

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000).fit(train_emb, train_labels)
bert_acc = clf.score(test_emb, test_labels)
print(f"BERT + head accuracy: {bert_acc:.1%}")

## 12) LSTM vs. BERT

In [ ]:
print(f"From-scratch LSTM: {lstm_acc:.1%}")
print(f"Pretrained BERT:   {bert_acc:.1%}")

## Takeaway

With only a few thousand reviews, the **pretrained BERT** usually beats the
from-scratch LSTM -- it already learned English elsewhere and only needed a small
push. That's the power of **transfer learning**.

### *Contributed By: Sattam Altwaim*